# Apply TMD maps to plane, sphere, cube (reduced)

This is a compact notebook to apply one TMD capture to three templates:
- `plane`
- `sphere`
- `cube`

It uses `apply_maps_to_mesh` directly, writes outputs under `notebook/real/exports/tmd_apply_mesh_test/reduced`, and prints a concise summary.

In [ ]:
from pathlib import Path

from tmd.cli.commands.model import apply_maps_to_mesh
def truemap_repo_root(start: Path | None = None) -> Path:
    """Resolve TrueMapData repo root from cwd (works from repo root or notebooks/)."""
    p = (start or Path.cwd()).resolve()
    for d in (p, *p.parents):
        if (d / "examples").is_dir() or (d / "tmd").is_dir():
            return d
        if (d / ".git").is_dir():
            return d
    if p.name == "notebooks":
        return p.parent
    return p

REPO_ROOT = truemap_repo_root()
TMD_FILE = REPO_ROOT / "examples" / "gelsight" / "circle_0mm_100g_heightmap_linear_detrend.tmd"
OUTPUT_BASE = REPO_ROOT / "notebooks" / "real" / "exports" / "tmd_apply_mesh_test" / "reduced"

# Keep your requested sizing defaults, but cap to make this notebook responsive.
OBJ_UNITS_TO_MM = 1000.0
MAX_TEXTURE_EDGE = 4096

if not TMD_FILE.is_file():
    raise FileNotFoundError(TMD_FILE)
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print("TMD:", TMD_FILE)
print("Output:", OUTPUT_BASE)


In [ ]:
jobs = [
    ("plane", "plane"),
    ("sphere", "sphere"),
    ("cube", "cube"),
]

results = {}
for template_kind, output_prefix in jobs:
    out_dir = OUTPUT_BASE / template_kind
    out_dir.mkdir(parents=True, exist_ok=True)

    result = apply_maps_to_mesh(
        tmd_file=TMD_FILE,
        output_root=out_dir,
        template_kind=template_kind,
        output_prefix=output_prefix,
        application_mode="uv",
        uv_alignment_mode="preserve",
        compress=75,
        normalize=True,
        obj_units_to_mm=OBJ_UNITS_TO_MM,
        tmd_mm_per_pixel=None,     # metadata-first
        max_texture_edge=MAX_TEXTURE_EDGE,
    )
    results[template_kind] = result

    print(f"[{template_kind}]")
    print("  obj:", result["obj"])
    print("  mtl:", result["mtl"])
    print("  textures:", result["textures_dir"])
    print("  target:", result["target_size_px"], "tile:", result["tile_size_px"], "cap:", result.get("scale_cap"))

In [ ]:
results